In [13]:
import os, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score, confusion_matrix

torch.manual_seed(42)
sp = pd.read_csv("data/splits.csv")
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
aug = transforms.Compose([transforms.RandomAffine(8, translate=(0.05, 0.05), scale=(0.95, 1.05)),
                          transforms.ToTensor(), norm])
plain = transforms.Compose([transforms.ToTensor(), norm])

class XR(Dataset):
    def __init__(self, d, tf): self.d, self.tf = d.reset_index(drop=True), tf
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        img = Image.open(f"data/processed/{r.patientId}.png").convert("RGB")
        return self.tf(img), torch.tensor(float(r.label))

mk = lambda s, tf, sh: DataLoader(XR(sp[sp.split == s], tf), batch_size=16, shuffle=sh)
tr_dl, va_dl, te_dl = mk("train", aug, True), mk("val", plain, False), mk("test", plain, False)
print({s: int((sp.split == s).sum()) for s in ["train", "val", "test"]})

{'train': 420, 'val': 60, 'test': 120}


In [14]:
m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
for p in m.parameters(): p.requires_grad = False
for p in list(m.features.denseblock4.parameters()) + list(m.features.norm5.parameters()):
    p.requires_grad = True
m.classifier = nn.Linear(1024, 1)
opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=1e-4)
lossf = nn.BCEWithLogitsLoss()

def predict(dl):
    m.eval(); ps, ys = [], []
    with torch.no_grad():
        for x, y in dl:
            ps += torch.sigmoid(m(x).squeeze(1)).tolist(); ys += y.tolist()
    return np.array(ps), np.array(ys)

In [15]:
os.makedirs("models", exist_ok=True)
best = 0
for ep in range(1, 9):
    m.train(); tot = 0
    for x, y in tr_dl:
        opt.zero_grad(); l = lossf(m(x).squeeze(1), y); l.backward(); opt.step(); tot += l.item()
    p, y = predict(va_dl)
    auc = roc_auc_score(y, p) if len(set(y)) > 1 else 0
    print(f"epoch {ep}  train loss {tot/len(tr_dl):.3f}  val AUC {auc:.3f}")
    if auc >= best: best = auc; torch.save(m.state_dict(), "models/densenet_pneumonia.pt")

epoch 1  train loss 0.592  val AUC 0.829
epoch 2  train loss 0.516  val AUC 0.817
epoch 3  train loss 0.503  val AUC 0.827
epoch 4  train loss 0.445  val AUC 0.834
epoch 5  train loss 0.416  val AUC 0.833
epoch 6  train loss 0.379  val AUC 0.829
epoch 7  train loss 0.340  val AUC 0.811
epoch 8  train loss 0.326  val AUC 0.821


In [16]:
m.load_state_dict(torch.load("models/densenet_pneumonia.pt"))
p, y = predict(te_dl)
tn, fp, fn, tp = confusion_matrix(y, p >= 0.5).ravel()
print("test AUC:", round(roc_auc_score(y, p), 3), "| accuracy:", round((tp + tn) / len(y), 3))
print("sensitivity (pneumonia caught):", round(tp / (tp + fn), 3), "| specificity (normal kept):", round(tn / (tn + fp), 3))
print("confusion [tn fp fn tp]:", tn, fp, fn, tp)

test AUC: 0.826 | accuracy: 0.742
sensitivity (pneumonia caught): 0.833 | specificity (normal kept): 0.65
confusion [tn fp fn tp]: 39 21 10 50
